# El detective de una ciudad: encontrar días extraños

## Ejercicio sencillo de analítica avanzada con Python y Gradio

En este ejercicio analizaremos el uso diario de un sistema de bicicletas compartidas. La pregunta es fácil de entender:

> **¿Qué días se comportaron de forma diferente a lo normal?**

El participante subirá un archivo CSV, lo explorará y construirá una pequeña herramienta que indique si una fecha parece normal o extraña. El objetivo es aprender un flujo completo: datos → modelo → visualización → aplicación interactiva.

## 1. ¿Qué problema resolvemos?

Una empresa de bicicletas necesita saber qué días merecen una revisión. Un día puede tener muy pocos alquileres por lluvia, o muchísimos por un evento. También puede ocurrir una combinación poco común: temperatura agradable, pero demanda inesperadamente baja.

En lugar de revisar manualmente cientos de días, construiremos un detector de anomalías. Una anomalía no significa que el dato esté mal; significa que el comportamiento fue diferente y conviene investigarlo.

### Dataset

Usaremos el archivo `day.csv` del **Bike Sharing Dataset** de la UCI. Contiene una fila por día y variables como fecha, alquileres, temperatura, humedad, viento, día laboral y clima. El archivo debe ser descargado previamente desde la fuente indicada y subido en la siguiente celda.

## 2. Preparar Colab

In [ ]:
# Instalamos las librerías que usaremos.
# Esta celda se ejecuta una sola vez en Google Colab.
!pip -q install gradio scikit-learn plotly seaborn

In [ ]:
# Importamos herramientas para leer datos, construir modelos y hacer gráficos.
import io
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Entorno preparado.")

## 3. Subir el dataset

Descarga `day.csv` desde el repositorio de la UCI: [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). Después pulsa **Choose Files** y selecciona el archivo.

El notebook no descarga datos automáticamente: el participante practica el paso real de recibir y validar un archivo.

In [ ]:
# El participante selecciona el archivo desde su computadora.
uploaded = files.upload()
file_name = next(iter(uploaded))
daily = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"Archivo recibido: {file_name}")
print(f"Filas: {daily.shape[0]:,} | Columnas: {daily.shape[1]}")
display(daily.head())

### Validar que el archivo sea el correcto

In [ ]:
# Estas columnas son las mínimas que necesitamos para el ejercicio.
required_columns = ["dteday", "cnt", "temp", "hum", "windspeed", "workingday", "holiday", "weathersit"]
missing_columns = [column for column in required_columns if column not in daily.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}. Sube el archivo day.csv original.")

print("Validación correcta: el archivo tiene las columnas necesarias.")
display(daily[required_columns].isna().sum().to_frame("valores faltantes"))

## 4. Preparar variables fáciles de interpretar

In [ ]:
# Convertimos la fecha y creamos etiquetas que luego usaremos en tablas y gráficos.
daily["fecha"] = pd.to_datetime(daily["dteday"])
daily["temperatura_c"] = daily["temp"] * 41
daily["tipo_dia"] = np.where(daily["workingday"] == 1, "Laboral", "No laboral")
daily["clima"] = daily["weathersit"].map({1: "Despejado", 2: "Nublado", 3: "Lluvia ligera", 4: "Lluvia intensa"})

print(f"Periodo: {daily['fecha'].min().date()} a {daily['fecha'].max().date()}")
display(daily[["fecha", "cnt", "temperatura_c", "tipo_dia", "clima"]].head())

La temperatura original está normalizada en una escala de 0 a 1. La convertimos aproximadamente a grados Celsius para que el resultado sea más fácil de leer. No cambia el patrón del análisis; solamente cambia la forma de presentar el dato.

## 5. Exploración visual

In [ ]:
# Gráfico 1: evolución de los alquileres a lo largo del tiempo.
fig = px.line(daily, x="fecha", y="cnt", title="Alquileres diarios",
              labels={"fecha": "Fecha", "cnt": "Número de alquileres"})
fig.update_layout(template="plotly_white", height=420)
fig.show()

In [ ]:
# Gráfico 2: comparamos días laborales y no laborales.
plt.figure(figsize=(8, 4))
sns.boxplot(data=daily, x="tipo_dia", y="cnt")
plt.title("Distribución de alquileres según el tipo de día")
plt.xlabel("")
plt.ylabel("Alquileres diarios")
plt.show()

In [ ]:
# Gráfico 3: observamos la relación entre temperatura y demanda.
fig = px.scatter(daily, x="temperatura_c", y="cnt", color="clima",
                 hover_data=["fecha", "tipo_dia"], size="cnt",
                 title="Temperatura, clima y alquileres",
                 labels={"temperatura_c": "Temperatura aproximada (°C)", "cnt": "Alquileres"})
fig.update_layout(template="plotly_white", height=450)
fig.show()

Los gráficos nos ayudan a conocer el terreno antes de modelar. Vemos tendencia, diferencias entre tipos de día y la relación entre clima y demanda. Esta exploración evita tratar los datos como una caja negra.

## 6. Detectar días extraños con Isolation Forest

Isolation Forest puede explicarse como un juego de “aislar puntos”. Si un día es muy diferente, suele quedar separado del resto con pocos cortes. El algoritmo repite este proceso muchas veces y asigna una puntuación.

Antes de modelar, estandarizamos las variables. Esto pone alquileres, humedad y viento en escalas comparables. La opción `contamination=0.05` indica que esperamos aproximadamente un 5% de días atípicos; es un parámetro didáctico que se podría ajustar en un proyecto real.

In [ ]:
# Seleccionamos las variables que describen el comportamiento de un día.
features = ["cnt", "temp", "hum", "windspeed", "workingday", "holiday", "weathersit"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(daily[features])

model = IsolationForest(n_estimators=250, contamination=0.05, random_state=RANDOM_STATE)
model.fit(X_scaled)

# Isolation Forest devuelve -1 para anomalía y 1 para día normal.
daily["es_anomalia"] = model.predict(X_scaled) == -1
daily["puntuacion_anomalia"] = -model.score_samples(X_scaled)
daily["resultado"] = np.where(daily["es_anomalia"], "Anómalo", "Normal")

print(f"Días anómalos: {daily['es_anomalia'].sum()} de {len(daily)} ({daily['es_anomalia'].mean():.1%})")
display(daily.nlargest(10, "puntuacion_anomalia")[["fecha", "cnt", "tipo_dia", "clima", "puntuacion_anomalia", "resultado"]])

In [ ]:
# Visualizamos dónde aparecen las anomalías en el calendario.
fig = px.scatter(daily, x="fecha", y="cnt", color="resultado",
                 color_discrete_map={"Normal": "#94a3b8", "Anómalo": "#ef4444"},
                 hover_data=["puntuacion_anomalia", "tipo_dia", "clima"],
                 title="Días normales y anómalos",
                 labels={"fecha": "Fecha", "cnt": "Alquileres", "resultado": "Clasificación"})
fig.update_layout(template="plotly_white", height=470)
fig.show()

Un día rojo no es necesariamente un error. Es una alerta para preguntar qué pasó. Por ejemplo, puede haber una tormenta, un evento, un día festivo o una falla de disponibilidad.

### ¿Qué hace cada parte del código?

- `features` define qué información usará el modelo. Incluimos demanda, clima y calendario porque juntos describen el contexto del día.
- `StandardScaler` coloca las variables en una escala comparable. Así, una variable no domina solo porque sus números sean más grandes.
- `IsolationForest` aprende el patrón general sin que tengamos que etiquetar antes los días.
- `n_estimators=250` crea muchos árboles aleatorios para que el resultado sea más estable.
- `contamination=0.05` pide al modelo que señale aproximadamente el 5% de los días como casos atípicos.
- `predict` produce la etiqueta normal/anómalo.
- `score_samples` produce una medida continua. En este notebook la invertimos para que un número mayor sea más fácil de interpretar como “más extraño”.

## 7. Interpretar los resultados

La tabla de resultados debe leerse como una lista de prioridades para investigar.

- **Normal:** el día se parece al comportamiento aprendido.
- **Anómalo:** el día se aleja del patrón en la combinación de variables analizadas.
- **Puntuación de anomalía:** permite ordenar los casos. No es un porcentaje de probabilidad ni una medida de impacto económico.

Un resultado alto no explica por sí mismo la causa. La causa puede estar fuera del archivo: un concierto, una avería, un cierre vial o un problema de registro. Por eso revisaremos qué variables estuvieron más lejos de lo habitual.

## 8. Explicar una anomalía con variables originales

In [ ]:
# Comparamos cada día con la mediana de cada variable.
# Una desviación mayor indica que el valor está más alejado de lo habitual.
medians = daily[features].median()
mad = (daily[features] - medians).abs().median().replace(0, 1e-9)

def explain_day(row, top_n=4):
    deviations = ((row[features] - medians).abs() / mad).sort_values(ascending=False)
    explanation = []
    for feature in deviations.head(top_n).index:
        direction = "por encima" if row[feature] > medians[feature] else "por debajo"
        explanation.append({
            "variable": feature,
            "dirección": direction,
            "valor del día": round(float(row[feature]), 3),
            "mediana": round(float(medians[feature]), 3),
            "desviación relativa": round(float(deviations[feature]), 2)
        })
    return pd.DataFrame(explanation)

most_unusual = daily.nlargest(1, "puntuacion_anomalia").iloc[0]
print(f"Ejemplo de explicación para {most_unusual['fecha'].date()}:")
display(explain_day(most_unusual))

La explicación compara cada variable con la **mediana**, que representa un día típico. Usamos también una medida robusta de dispersión para no dejar que unos pocos valores extremos distorsionen la comparación.

Por ejemplo, si `cnt` aparece por encima y con una desviación relativa alta, significa que ese día tuvo muchos más alquileres de lo habitual. Si `hum` aparece por debajo, la humedad fue menor que el valor típico. El objetivo es construir una explicación que una persona pueda leer sin conocer todos los detalles matemáticos.

La explicación identifica señales importantes, pero no demuestra que una variable haya causado la anomalía. Para confirmar la historia habría que consultar información de eventos, clima detallado y operación del sistema.

## 9. Interfaz Gradio

La interfaz permite que alguien use el análisis sin modificar el código.

1. El menú desplegable muestra todas las fechas disponibles en el archivo subido.
2. Al pulsar **Analizar**, se busca esa fecha en el dataframe ya preparado.
3. La función devuelve un reporte en Markdown con el estado, el contexto y la puntuación.
4. La tabla muestra las variables más diferentes respecto al comportamiento típico.
5. El gráfico coloca la fecha seleccionada sobre toda la serie, para evitar interpretar el dato sin contexto.

El evento `button.click(...)` conecta el botón con la función. El evento `date_input.change(...)` hace que el resultado también se actualice al cambiar la fecha. Gradio se encarga de presentar estos resultados en una página web temporal.

In [ ]:
# Función que crea el gráfico de contexto para una fecha seleccionada.
def context_plot(selected_date):
    row = daily.loc[daily["fecha"] == pd.to_datetime(selected_date)].iloc[0]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daily["fecha"], y=daily["cnt"], mode="lines",
                             name="Todos los días", line=dict(color="#cbd5e1", width=1)))
    fig.add_trace(go.Scatter(x=[row["fecha"]], y=[row["cnt"]], mode="markers",
                             name="Fecha elegida", marker=dict(size=16, color="#f97316")))
    fig.update_layout(template="plotly_white", height=360, title="Contexto de la fecha")
    return fig

def analyze_date(selected_date):
    row = daily.loc[daily["fecha"] == pd.to_datetime(selected_date)].iloc[0]
    status = "🔴 DÍA ANÓMALO" if row["es_anomalia"] else "🟢 DÍA NORMAL"
    report = f"""## {status}

**Fecha:** {row['fecha'].strftime('%d/%m/%Y')}  
**Alquileres:** {row['cnt']:,}  
**Tipo de día:** {row['tipo_dia']}  
**Clima:** {row['clima']}  
**Temperatura aproximada:** {row['temperatura_c']:.1f} °C  
**Puntuación relativa:** {row['puntuacion_anomalia']:.3f}

Las variables siguientes fueron las más diferentes de su comportamiento habitual."""
    table = explain_day(row).rename(columns={"variable": "Variable", "dirección": "Dirección"})
    return report, table, context_plot(selected_date)

import gradio as gr
date_choices = [date.strftime("%Y-%m-%d") for date in daily["fecha"].sort_values()]

custom_css = """
body { background: #f8fafc; }
.gradio-container { max-width: 1050px !important; }
#hero { background: linear-gradient(135deg, #0f172a, #1e3a5f); color: white; padding: 24px; border-radius: 18px; margin-bottom: 16px; }
#hero h1 { margin: 0 0 8px 0; }
.note { color: #475569; font-size: 14px; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange"), title="Días extraños") as demo:
    gr.HTML("<div id='hero'><h1>🔎 El detective de una ciudad</h1><p>Investiga si una fecha tuvo un comportamiento normal o atípico.</p></div>")
    gr.Markdown("Selecciona una fecha para consultar el resultado del modelo.", elem_classes=["note"])
    with gr.Row():
        date_input = gr.Dropdown(choices=date_choices, value=date_choices[-1], label="Fecha", scale=3)
        button = gr.Button("Analizar", variant="primary", scale=1)
    report = gr.Markdown()
    with gr.Row():
        table = gr.Dataframe(label="Variables más diferentes", interactive=False)
        plot = gr.Plot(label="Contexto temporal")
    gr.Markdown("*Una anomalía es una señal estadística, no una prueba de causalidad.*", elem_classes=["note"])
    button.click(analyze_date, inputs=date_input, outputs=[report, table, plot])
    date_input.change(analyze_date, inputs=date_input, outputs=[report, table, plot])

demo.launch(share=True)

## 10. Conclusiones

- Aprendimos a recibir y validar un dataset subido por otra persona.
- Exploramos la demanda con gráficos antes de construir el modelo.
- Isolation Forest encontró días cuyo comportamiento merece investigación.
- La explicación por variables originales ayuda a comunicar el resultado en lenguaje sencillo.
- Gradio convirtió el análisis en una herramienta que puede usar alguien sin escribir código.

### Preguntas para discusión

1. ¿Qué día investigarías primero y por qué?
2. ¿Qué información adicional pedirías para explicar una anomalía?
3. ¿Qué cambiaría si el costo de una falsa alerta fuera muy alto?
4. ¿El porcentaje de anomalías del 5% te parece razonable para este caso?